# Wizualizacja danych genomicznych

## Wprowadzenie

W tym notatniku przeprowadzimy analizę i wizualizację cech sekwencji DNA, takich jak zawartość GC i asymetria GC, wykorzystując technikę przesuwnych okien. Następnie dane zwizualizujemy przy pomocy [Circos](http://circos.ca/) - popularnego narzędzia umożliwiającego kolistą reprezentację danych genomicznych. Pakiet Circos jest niezwykle elastyczny i doskonale udokumentowany, [z wieloma przykładami i dokładnymi opisami funkcjonalności.](http://circos.ca/documentation/) 
![http://circos.ca/intro/genomic_data/](https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41588-021-00922-y/MediaObjects/41588_2021_922_Fig1_HTML.png)
*https://doi.org/10.1038/s41588-021-00922-y*

## Import Bibliotek

Przed rozpoczęciem analizy danych i wizualizacji importujemy kilka kluczowych bibliotek:
1. sys: Umożliwia interakcję z interpreterem Pythona i zarządzanie argumentami wiersza poleceń.
2. pandas (pd): Służy do łatwego wczytywania i przetwarzania danych w formie tabelarycznej.
3. numpy (np): Zawiera narzędzia do obliczeń numerycznych i operacji na danych wielowymiarowych.
4. plotnine: Umożliwia tworzenie wykresów i wizualizację danych, oparta jest na gramatyce pakietu ggplot2 z języka R.

In [1]:
import sys
import pandas as pd
import numpy as np
from plotnine import *

## Zdefiniowanie funkcji 

Zdefinujmy funkcje wykorzystywane podczas analizy

#### Odczytywanie Sekwencji DNA


Aby analizować sekwencje DNA, musimy najpierw załadować dane z pliku FASTA.

Funkcja *read_fasta* służy do wczytywania sekwencji DNA z pliku w formacie FASTA do słownika. 

**Parametry:**
*file_path:* Ścieżka do pliku FASTA z sekwencją DNA.


**Zwraca:**
*fadict:* Słownik, w którym kluczami są nazwy sekwencji, a wartościami są odpowiadające im sekwencje DNA.


In [2]:
def read_fasta(file_path):
    fadict = {}
    with open(file_path) as file:
        idx = ""
        for line in file:
            if line.startswith(">"):
                idx = line.strip().split()[0][1:]
                fadict[idx] = ""
            else:
                fadict[idx] += line.strip()
    return fadict

#### Analiza w ruchomych oknach

Jedną z podstawowych technik analizy sekwencji jest analiza w ruchomych oknach (sliding windows). Polega ona na podziale sekwencji na małe, nakładające się segmenty (okna) i wykonywaniu obliczeń w każdym z nich.

Funkcja *sliding_window* wykonuje analizę ruchomych okien na sekwencji DNA.

**Parametry:**
*seq:* Analizowana sekwencja DNA.
*size:* Rozmiar okna.
*step:* Krok przesuwania okna.
*func:* Funkcja, która ma zostać zastosowana do każdego okna.

**Zwraca:**
Lista wyników analizy okien, gdzie każdy wynik zawiera informacje o początku i końcu okna oraz wynik funkcji *func* na tym oknie.


In [3]:
def sliding_window(seq, size, step, func):
    results = []
    for start in range(0, len(seq) - size + 1, step):
        subseq = seq[start:start + size]
        results.append([start, start + size, func(subseq)])
    return results

#### Obliczanie Zawartości GC

**Wprowadzenie:**
Zawartość GC to miara proporcji zasad guaniny (G) i cytozyny (C) w sekwencji DNA. 

Funkcja *gc_content* Oblicza zawartość GC w danej sekwencji DNA.

**Parametry:**
*seq:* Analizowana sekwencja DNA.

**Zwraca:**
Procentowy udział GC w sekwencji.


In [4]:

def gc_content(seq):
    gc_count = sum(1 for base in seq if base.lower() in ["g", "c"])
    return gc_count / len(seq)

#### Obliczanie Skosu GC

Asymetria GC (GC skew) określa różnicę w zawartości guaniny i cytozyny na jednej nici DNA. 

Funkcja *gc_skew* oblicza asymetrię GC w danej sekwencji DNA.
**Parametry:**

*seq:* Analizowana sekwencja DNA.

**Zwraca:**
Asymetrię GC w sekwencji.


In [5]:
def gc_skew(seq):
    g = seq.lower().count("g")
    c = seq.lower().count("c")
    return (g - c) / (g + c) if (g + c) > 0 else 0 

#### Skalowanie Min-Max

W niektórych przypadkach ważne jest przeskalowanie naszych danych tak, aby mieściły się w określonym zakresie, na przykład [0, 1].

Funkcja *rescale_range* wykonuje przeskalowanie wartości x z zakresu [min_x, max_x] na inny zakres [i, j].

**Parametry:**
*x:* Wartość do przeskalowania.
*min_x:* Dolny zakres oryginalnych wartości.
*max_x:* Górny zakres oryginalnych wartości.
*i:* Dolny zakres przeskalowanych wartości.
*j:* Górny zakres przeskalowanych wartości.

**Zwraca:**
Przeskalowaną wartość *x* w nowym zakresie [i, j].


In [6]:
def rescale_range(x, min_x, max_x, i=0, j=1):
    return(i+((x-min_x)*(j-i)/(max_x-min_x)))

#### Analiza Częstości K-merów

K-mery to sekwencje o długości k w sekwencji DNA. Analiza częstości k-merów może dostarczyć wglądu w charakterystykę sekwencji. 

Funkcja *kmer_freq* analizuje sekwencję DNA w poszukiwaniu wystąpień k-merów i zlicza ich częstość.

**Parametry:**
*seq:* Analizowana sekwencja DNA.
*k:* Długość k-mera (domyślnie 20).

**Zwraca:**
Słownik, w którym kluczami są k-mery, a wartościami liczba ich wystąpień w sekwencji.

In [7]:
def kmer_freq(seq, k=20):
    kmer_dic = {}
    for i in range(len(seq) - k + 1):
        subseq = seq[i:i + k]
        kmer_dic[subseq] = kmer_dic.get(subseq, 0) + 1
    return kmer_dic

#### Pobieranie Pozycji K-merów

Funkcja *kmer_pos* znajduje pozycje wystąpień danego k-mera w sekwencji DNA.

**Parametry:**
*seq:* Analizowana sekwencja DNA.
*kmer:* K-mer, którego pozycje chcemy znaleźć.

**Zwraca:**
Lista pozycji wystąpień k-mera, gdzie każda pozycja to krotka (start, stop).


In [8]:
def kmer_pos(seq, kmer):
    pos = []
    k = len(kmer)
    start = 0
    while True:
        start = seq.find(kmer, start)
        if start == -1:
            break
        pos.append((start, start + k))
        start += 1
    return pos


#### Scalanie Nachodzących Zakresów 

Funkcja merge_overlaps łączy nakładające się zakresy (pary start-stop) w sekwencjach. Funkcja wymaga posortowanych zakresów wejściowych. 

**Parametry:**
*ranges:* Lista zakresów do połączenia.

**Zwraca:**
Złączone zakresy, które eliminują nakładające się fragmenty i tworzą spójne obszary.


In [9]:
def merge_overlaps(ranges):
    if not ranges:
        return []
    
    ranges = sorted(ranges)
    merged = [ranges[0]]
    
    for current in ranges[1:]:
        last = merged[-1]
        if current[0] <= last[1]:  # Overlapping intervals
            merged[-1] = [last[0], max(last[1], current[1])]
        else:
            merged.append(current)
    return merged


## Przygotowanie genomu Wejściowego

W tej sekcji przygotujemy dane genomu do analizy. Pobierzemy i rozpakujemy plik FASTA z sekwencją genomu i zmienimy jego nazwę na bardziej intuicyjną. Proszę zmienić genom na wybrany przez siebie kompletny genom bakteryjny, opublikowany nie wcześniej niż w 2018 roku. Genom można pobrać z bazy danych [NCBI Genomes](https://www.ncbi.nlm.nih.gov/datasets/genomes/). 


In [12]:
!wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/027/325/GCF_000027325.1_ASM2732v1/GCF_000027325.1_ASM2732v1_genomic.fna.gz

zsh:1: command not found: wget


In [13]:
!gunzip GCF_000027325.1_ASM2732v1_genomic.fna.gz 

gunzip: can't stat: GCF_000027325.1_ASM2732v1_genomic.fna.gz (GCF_000027325.1_ASM2732v1_genomic.fna.gz.gz): No such file or directory


In [14]:
!mv GCF_000027325.1_ASM2732v1_genomic.fna mycoplasma.fa

mv: rename GCF_000027325.1_ASM2732v1_genomic.fna to mycoplasma.fa: No such file or directory


In [10]:
in_fasta = "GENOME_thin.fasta"
fadict = read_fasta(in_fasta)

Analizę przeprowadzimy jedynie dla chromosomu bakteryjnego, więc jeżeli w złożeniu znajdują się jakiekolwiek dodatkowe sekwencje (np. plazmidy) proszę je zignorować, i do dalszych kroków wykorzystać jedynie sekwencję największej cząsteczki.

In [11]:
fadict.keys()

dict_keys(['ptg000007l', 'ptg000003l', 'ptg000012l', 'ptg000008l', 'ptg000002l', 'ptg000016l', 'ptg000018l', 'ptg000014l', 'ptg000011l', 'ptg000015l', 'ptg000010l', 'ptg000004l', 'ptg000001l', 'ptg000009l', 'ptg000013l', 'ptg000017l'])

In [12]:
for key in fadict.keys():
    print(key, len(fadict[key]))

ptg000007l 4962671
ptg000003l 1805757
ptg000012l 2044213
ptg000008l 2324804
ptg000002l 2259285
ptg000016l 2349775
ptg000018l 2524214
ptg000014l 2808376
ptg000011l 3109985
ptg000015l 3330951
ptg000010l 4100071
ptg000004l 3736295
ptg000001l 3711849
ptg000009l 4221094
ptg000013l 4262825
ptg000017l 4416379


In [13]:
chromnames = list(fadict.keys())
chromnames

['ptg000007l',
 'ptg000003l',
 'ptg000012l',
 'ptg000008l',
 'ptg000002l',
 'ptg000016l',
 'ptg000018l',
 'ptg000014l',
 'ptg000011l',
 'ptg000015l',
 'ptg000010l',
 'ptg000004l',
 'ptg000001l',
 'ptg000009l',
 'ptg000013l',
 'ptg000017l']

In [14]:
for chromname in chromnames:
    seq = fadict[chromname]
    seq_length = len(seq)
    print(seq_length)

4962671
1805757
2044213
2324804
2259285
2349775
2524214
2808376
3109985
3330951
4100071
3736295
3711849
4221094
4262825
4416379


### Analiza Zawartości GC z Wykorzystaniem Okien


W tej sekcji wykorzystamy funkcję *sliding_window*, aby obliczyć zawartość GC w sekwencji genomu korzystając z okien o zdefiniowanej wielkości i kroku. Poeksperymentuj z wielkością okna oraz kroku, aby zobaczyć jak parametry te wpływają na wynik. Poniważ Circos wymaga, aby zakresy danych nie nakładały się, w ostatnim przebiegu ustaw długość kroku równą wielkości okna.


In [ ]:
w_size = 10000
w_step = 10000

In [ ]:
from copy import deepcopy
gcdf_dict = {}

for chromname in chromnames:
    seq = fadict[chromname]
    seq_length = len(seq)
    print(seq_length)
    
    gc = sliding_window(seq, w_size, w_step, gc_content)

    gcdf = pd.DataFrame(gc, columns=["start", "stop", "gc"])
    print(gcdf)
    gcdf_dict[chromname]= gcdf

print(len(gcdf_dict))


4962671
       start     stop      gc
0          0    10000  0.4670
1      10000    20000  0.4690
2      20000    30000  0.4730
3      30000    40000  0.4639
4      40000    50000  0.5122
..       ...      ...     ...
491  4910000  4920000  0.4237
492  4920000  4930000  0.4492
493  4930000  4940000  0.4675
494  4940000  4950000  0.4239
495  4950000  4960000  0.4394

[496 rows x 3 columns]
1805757
       start     stop      gc
0          0    10000  0.5104
1      10000    20000  0.5020
2      20000    30000  0.5740
3      30000    40000  0.5047
4      40000    50000  0.4715
..       ...      ...     ...
175  1750000  1760000  0.4572
176  1760000  1770000  0.4997
177  1770000  1780000  0.4750
178  1780000  1790000  0.4793
179  1790000  1800000  0.4829

[180 rows x 3 columns]
2044213
       start     stop      gc
0          0    10000  0.5019
1      10000    20000  0.4458
2      20000    30000  0.5122
3      30000    40000  0.4837
4      40000    50000  0.4998
..       ...      ...     ..

#### Zawartość GC z poziomu terminala

Do szybkiego obliczenia zawartości GC możemy wykorzystać narzędzie **fx2tab** z pakietu **seqkit** korzystając z flag **-n -B GC**. Argumenty flagi **-B** określają nukleotydy, których relatywny udział w sekwencji chcemy określić. Alternatywnie możemy posłużyć się równoważnym **-n -g**

In [ ]:
!seqkit fx2tab {in_fasta} -n -B GC

NC_000908.2 Mycoplasmoides genitalium G37, complete sequence	31.69


#### Seqkit umożliwia również analizę w oknie

In [ ]:
! cat {in_fasta}| \
 seqkit sliding -s 5000 -W 5000 | \
 seqkit fx2tab -n -g | head

NC_000908.2_sliding:1-5000	28.28
NC_000908.2_sliding:5001-10000	30.54
NC_000908.2_sliding:10001-15000	26.64
NC_000908.2_sliding:15001-20000	29.14
NC_000908.2_sliding:20001-25000	29.52
NC_000908.2_sliding:25001-30000	29.06
NC_000908.2_sliding:30001-35000	30.88
NC_000908.2_sliding:35001-40000	30.32
NC_000908.2_sliding:40001-45000	30.24
NC_000908.2_sliding:45001-50000	31.52


### Wizualizacja Zawartości GC

Dane z poprzednich sekcji zostaną użyte do utworzenia wykresu liniowego, który pokaże zmiany zawartości GC w sekwencji genomu.


In [ ]:
for chromname, gcdf in gcdf_dict.items():
    gcplot = (ggplot(gcdf, aes(x="start", y="gc")) + geom_line() + theme_minimal())
    print(gcplot)
    gcplot.save(f"gc_plot_{chromname}.png", width=8, height = 6)
    

<ggplot: (640 x 480)>


/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000007l.png


<ggplot: (640 x 480)>


/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000003l.png


<ggplot: (640 x 480)>


...

/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000009l.png


<ggplot: (640 x 480)>


/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000013l.png


<ggplot: (640 x 480)>


/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000017l.png


### Sformatowanie danych do narzędzia Circos

Dane dotyczące zawartości GC zwizualizujemy w postaci histogramu. W tym celu Circos wymaga danych w formacie (bez nagłówka):

|chromosom|start|stop|wartość|kolor|
|--|--|--|--|--|
|Chr1|	0	|100|	1.629407336396254|	fill_color=red|
|Chr1	|100|	200|	1.5523160283080437|	fill_color=red|


#### Dodanie Informacji o Chromosomie


W tej sekcji przypiszemy do naszych danych identyfikator chromosomu. 

In [ ]:
gcdf_dict_insert = deepcopy(gcdf_dict)

for chromname, gcdf in gcdf_dict.items():
    gcdf.insert(0,"chr",chromname)
    gcdf_dict_insert[chromname] = gcdf
    print(gcdf)

            chr    start     stop      gc
0    ptg000007l        0    10000  0.4670
1    ptg000007l    10000    20000  0.4690
2    ptg000007l    20000    30000  0.4730
3    ptg000007l    30000    40000  0.4639
4    ptg000007l    40000    50000  0.5122
..          ...      ...      ...     ...
491  ptg000007l  4910000  4920000  0.4237
492  ptg000007l  4920000  4930000  0.4492
493  ptg000007l  4930000  4940000  0.4675
494  ptg000007l  4940000  4950000  0.4239
495  ptg000007l  4950000  4960000  0.4394

[496 rows x 4 columns]
            chr    start     stop      gc
0    ptg000003l        0    10000  0.5104
1    ptg000003l    10000    20000  0.5020
2    ptg000003l    20000    30000  0.5740
3    ptg000003l    30000    40000  0.5047
4    ptg000003l    40000    50000  0.4715
..          ...      ...      ...     ...
175  ptg000003l  1750000  1760000  0.4572
176  ptg000003l  1760000  1770000  0.4997
177  ptg000003l  1770000  1780000  0.4750
178  ptg000003l  1780000  1790000  0.4793
179  ptg00

#### Przypisanie Kolorów do Zakresów GC
W tej części przypiszemy kolory do różnych zakresów zawartości GC. W zależności od wartości zawartości GC, zakresy te otrzymają różne kolory na wykresie. Kolory określimy w formacie "R,G,B".


In [ ]:
gcdf_dict_final = deepcopy(gcdf_dict_insert)

for chromname, gcdf in gcdf_dict_insert.items():
    # Definiujemy przedziały (bins) oraz odpowiadające im etykiety kolorów
    bins = [0, 0.25, 0.35, float('inf')]
    colors = ["fill_color=103,201,129", "fill_color=201,193,103", "fill_color=204,116,92"]

    # Używamy pd.cut do zaklasyfikowania wartości w kolumnie 'gc' i przypisania odpowiednich etykiet na podstawie przedziałów
    gcdf['col'] = pd.cut(gcdf['gc'], bins=bins, labels=colors, right=True)
    gcdf_dict_final[chromname] = gcdf
    print(gcdf)

            chr    start     stop      gc                    col
0    ptg000007l        0    10000  0.4670  fill_color=204,116,92
1    ptg000007l    10000    20000  0.4690  fill_color=204,116,92
2    ptg000007l    20000    30000  0.4730  fill_color=204,116,92
3    ptg000007l    30000    40000  0.4639  fill_color=204,116,92
4    ptg000007l    40000    50000  0.5122  fill_color=204,116,92
..          ...      ...      ...     ...                    ...
491  ptg000007l  4910000  4920000  0.4237  fill_color=204,116,92
492  ptg000007l  4920000  4930000  0.4492  fill_color=204,116,92
493  ptg000007l  4930000  4940000  0.4675  fill_color=204,116,92
494  ptg000007l  4940000  4950000  0.4239  fill_color=204,116,92
495  ptg000007l  4950000  4960000  0.4394  fill_color=204,116,92

[496 rows x 5 columns]
            chr    start     stop      gc                    col
0    ptg000003l        0    10000  0.5104  fill_color=204,116,92
1    ptg000003l    10000    20000  0.5020  fill_color=204,116,92
2

#### Zapisywanie Danych

Teraz, gdy przeprowadziliśmy analizę zawartości GC, przypisaliśmy kolory i dodaliśmy informacje o chromosomie, zapiszemy te dane do pliku.

In [ ]:
for chromname, gcdf in gcdf_dict_final.items():
    gcdf.to_csv(f"gc_content_{chromname}.histo", sep="\t", index=False, header=False)

### Analiza asymetrii GC z Wykorzystaniem Okien

W tej sekcji przeprowadzimy analizę asymetrii GC. Tak jak w przypadku zawartości GC przeprowadźmy analizę dla różnych wartości długości okna oraz kroku.


In [ ]:
skew_data = {}

for chromname in chromnames:
    seq = fadict[chromname]
    #seq_length = len(seq)
    #print(seq_length)

    skew = sliding_window(seq, w_size, w_step, gc_skew)
    skewdf = pd.DataFrame(skew, columns=["start", "stop", "skew"])
    skew_data[chromname] = skewdf
    print(skewdf)

       start     stop      skew
0          0    10000  0.099786
1      10000    20000  0.046055
2      20000    30000 -0.016490
3      30000    40000 -0.039448
4      40000    50000 -0.033190
..       ...      ...       ...
491  4910000  4920000 -0.042247
492  4920000  4930000 -0.110864
493  4930000  4940000 -0.048556
494  4940000  4950000 -0.055438
495  4950000  4960000  0.000000

[496 rows x 3 columns]
       start     stop      skew
0          0    10000 -0.070141
1      10000    20000  0.050598
2      20000    30000 -0.042857
3      30000    40000 -0.016842
4      40000    50000 -0.045599
..       ...      ...       ...
175  1750000  1760000  0.014873
176  1760000  1770000 -0.014209
177  1770000  1780000  0.022316
178  1780000  1790000  0.031922
179  1790000  1800000 -0.024229

[180 rows x 3 columns]
       start     stop      skew
0          0    10000  0.039251
1      10000    20000 -0.161956
2      20000    30000  0.013667
3      30000    40000 -0.007236
4      40000    50000  0

#### Asymetria GC przy pomocy seqkit

In [ ]:
!cat {in_fasta}| \
seqkit sliding -s 5000 -W 5000 | \
seqkit fx2tab -n -G|head 

NC_000908.2_sliding:1-5000	10.47
NC_000908.2_sliding:5001-10000	11.33
NC_000908.2_sliding:10001-15000	1.80
NC_000908.2_sliding:15001-20000	7.89
NC_000908.2_sliding:20001-25000	11.92
NC_000908.2_sliding:25001-30000	13.15
NC_000908.2_sliding:30001-35000	-3.50
NC_000908.2_sliding:35001-40000	2.37
NC_000908.2_sliding:40001-45000	8.60
NC_000908.2_sliding:45001-50000	2.41


#### Wizualizacja asymetrii GC

Przygotowane dane asymetrii GC zostaną użyte do utworzenia wykresu liniowego.


In [ ]:
for chromname, skewdf in skew_data.items():
    skewplot = (ggplot(skewdf, aes(x="start", y="skew")) + geom_line()) + theme_minimal()
    skewplot.save(f"skew_plot_{chromname}", width=8, height=6)

/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_ptg000007l
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_ptg000003l
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_ptg000012l
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/

### Sformatowanie danych do narzędzia Circos

Dane dotyczące zawartości GC zwizualizujemy w postaci wykresu punktowego (scatterplot). Format danych dla tego typu wykresu jest tożsamy z powyższym, rodzaj wykresu określamy w pliku konfiguracyjnym, co zrobimy w dalszej części.


#### Dodawanie Informacji o Chromosomie


W tej sekcji przypiszemy do naszych danych identyfikator chromosomu. 

In [ ]:
skew_data_insert = deepcopy(skew_data)

for chromname, skewdf in skew_data.items():

    skewdf.insert(0,"chr",chromname)
    skew_data_insert[chromname] = skewdf
    print(skewdf)


            chr    start     stop      skew
0    ptg000007l        0    10000  0.099786
1    ptg000007l    10000    20000  0.046055
2    ptg000007l    20000    30000 -0.016490
3    ptg000007l    30000    40000 -0.039448
4    ptg000007l    40000    50000 -0.033190
..          ...      ...      ...       ...
491  ptg000007l  4910000  4920000 -0.042247
492  ptg000007l  4920000  4930000 -0.110864
493  ptg000007l  4930000  4940000 -0.048556
494  ptg000007l  4940000  4950000 -0.055438
495  ptg000007l  4950000  4960000  0.000000

[496 rows x 4 columns]
            chr    start     stop      skew
0    ptg000003l        0    10000 -0.070141
1    ptg000003l    10000    20000  0.050598
2    ptg000003l    20000    30000 -0.042857
3    ptg000003l    30000    40000 -0.016842
4    ptg000003l    40000    50000 -0.045599
..          ...      ...      ...       ...
175  ptg000003l  1750000  1760000  0.014873
176  ptg000003l  1760000  1770000 -0.014209
177  ptg000003l  1770000  1780000  0.022316
178  ptg

#### Przypisanie Kolorów do asymetrii GC
W tej części przypiszemy kolory do różnych zakresów asymetrii GC. Skorzystamy z domyślnie zdefiniowanych kolorów.

In [ ]:
skew_data_final = deepcopy(skew_data_insert)

for chromname, skewdf in skew_data_insert.items():
    skewdf["col"] = "fill_color=blue"
    skewdf.loc[skewdf["skew"]>0, "col"] = "fill_color=red"
    skew_data_final[chromname] = skewdf
    print(skewdf)

            chr    start     stop      skew              col
0    ptg000007l        0    10000  0.099786   fill_color=red
1    ptg000007l    10000    20000  0.046055   fill_color=red
2    ptg000007l    20000    30000 -0.016490  fill_color=blue
3    ptg000007l    30000    40000 -0.039448  fill_color=blue
4    ptg000007l    40000    50000 -0.033190  fill_color=blue
..          ...      ...      ...       ...              ...
491  ptg000007l  4910000  4920000 -0.042247  fill_color=blue
492  ptg000007l  4920000  4930000 -0.110864  fill_color=blue
493  ptg000007l  4930000  4940000 -0.048556  fill_color=blue
494  ptg000007l  4940000  4950000 -0.055438  fill_color=blue
495  ptg000007l  4950000  4960000  0.000000  fill_color=blue

[496 rows x 5 columns]
            chr    start     stop      skew              col
0    ptg000003l        0    10000 -0.070141  fill_color=blue
1    ptg000003l    10000    20000  0.050598   fill_color=red
2    ptg000003l    20000    30000 -0.042857  fill_color=blue


#### Zapisywanie danych

In [ ]:
for chromname, skewdf in skew_data_final.items():
    skewdf.to_csv(f"gc_skew_{chromname}.histo", sep="\t", index=False, header=False)

### Obliczanie Kumulatywnej Asymetrii GC

Kumulatywna asymetria GC to kumulatywna suma skosu GC kolejnych oknach. W tej sekcji obliczymy i przeskalujemy te wartości, aby uzyskać wyniki w określonym zakresie, co ułatwi nam zwizualizowanie asymetrii GC i kumulatywnej asymetrii GC na jednym wykresie.


In [ ]:
skew_data_cuml = deepcopy(skew_data_final)

for chromname, skewdf in skew_data_final.items():
    skew_cuml = []
    cumul = 0

    # Dla każdego okna dodajemy skew poprzedniego okna
    for idx in range(skewdf.shape[0]):
        cumul += skewdf.iloc[idx,]["skew"]
        skew_cuml.append(cumul)
    
    skew_cuml = np.array(skew_cuml)
    skew_data_cuml[chromname] = [skewdf, skew_cuml]

#### Przeskalowanie wartości kumulatywnej asymetrii GC do zakresu 0-1

In [ ]:
skew_data_cuml_scaled = deepcopy(skew_data_cuml)

for chromname, skew_df_cuml in skew_data_cuml.items():
    skewdf = skew_df_cuml[0]
    skew_cuml = skew_df_cuml[1]

    skew_cuml_scaled = rescale_range(skew_cuml, min(skew_cuml), max(skew_cuml))
    print(skew_cuml_scaled)
    skew_data_cuml_scaled[chromname] = [skewdf, skew_cuml_scaled]


[0.28046774 0.32735397 0.31056601 0.27040626 0.23661738 0.22040525
 0.24186561 0.26453368 0.29790168 0.26205076 0.19976614 0.17598446
 0.18188369 0.18772638 0.13071621 0.13624903 0.14345413 0.19382251
 0.25606946 0.25629953 0.25019787 0.21850528 0.14749995 0.11256725
 0.09635923 0.10481367 0.08092392 0.14022109 0.18119095 0.07162623
 0.         0.02983264 0.05946279 0.13852251 0.18704275 0.20887453
 0.21640952 0.26017184 0.2820561  0.27618318 0.3343087  0.37449989
 0.40399954 0.49486696 0.5105575  0.5213304  0.50660026 0.55107869
 0.59482094 0.61670739 0.61569603 0.64879803 0.67503826 0.68156676
 0.63586982 0.6060407  0.59040457 0.5766978  0.62293302 0.6563976
 0.5733056  0.57729617 0.53063519 0.63325351 0.73116825 0.86264232
 0.87774132 0.96445407 0.8845658  0.94011217 0.97113274 0.97378044
 0.9301502  0.90342364 1.         0.99955737 0.90803244 0.92323327
 0.88346987 0.87087967 0.92816409 0.87240228 0.80747804 0.70500538
 0.62068576 0.64413652 0.64963486 0.59224541 0.5762133  0.57712

#### Wizualizacja asymetrii GC i kumulatywnej asymetrii GC

W celu wizualizacji dołączymy kolumnę *skew_cuml* do ramki *skewdf*


In [ ]:
skew_data_final_scaled = {}

for chromname, skew_df_cuml in skew_data_cuml_scaled.items():
    skewdf = skew_df_cuml[0]
    skew_cuml_scaled = skew_df_cuml[1]

    skewdf["skew_cuml"] = skew_cuml_scaled
    print(skewdf)
    skew_data_final_scaled[chromname] = skewdf

            chr    start     stop      skew              col  skew_cuml
0    ptg000007l        0    10000  0.099786   fill_color=red   0.280468
1    ptg000007l    10000    20000  0.046055   fill_color=red   0.327354
2    ptg000007l    20000    30000 -0.016490  fill_color=blue   0.310566
3    ptg000007l    30000    40000 -0.039448  fill_color=blue   0.270406
4    ptg000007l    40000    50000 -0.033190  fill_color=blue   0.236617
..          ...      ...      ...       ...              ...        ...
491  ptg000007l  4910000  4920000 -0.042247  fill_color=blue   0.379409
492  ptg000007l  4920000  4930000 -0.110864  fill_color=blue   0.266545
493  ptg000007l  4930000  4940000 -0.048556  fill_color=blue   0.217113
494  ptg000007l  4940000  4950000 -0.055438  fill_color=blue   0.160676
495  ptg000007l  4950000  4960000  0.000000  fill_color=blue   0.160676

[496 rows x 6 columns]
            chr    start     stop      skew              col  skew_cuml
0    ptg000003l        0    10000 -0.070

#### Zmiana formatu z szerokiego (wide) na długi (long)

In [ ]:
skew_data_final_molten = {}

for chromname, skewdf in skew_data_final_scaled.items():
    skewdf_molten = pd.melt(skewdf, id_vars=["chr","start","stop"], value_vars=["skew","skew_cuml"])
    
    skew_data_final_molten[chromname] = skewdf_molten
    print(skewdf_molten)

            chr    start     stop   variable     value
0    ptg000007l        0    10000       skew  0.099786
1    ptg000007l    10000    20000       skew  0.046055
2    ptg000007l    20000    30000       skew -0.016490
3    ptg000007l    30000    40000       skew -0.039448
4    ptg000007l    40000    50000       skew -0.033190
..          ...      ...      ...        ...       ...
987  ptg000007l  4910000  4920000  skew_cuml  0.379409
988  ptg000007l  4920000  4930000  skew_cuml  0.266545
989  ptg000007l  4930000  4940000  skew_cuml  0.217113
990  ptg000007l  4940000  4950000  skew_cuml  0.160676
991  ptg000007l  4950000  4960000  skew_cuml  0.160676

[992 rows x 5 columns]
            chr    start     stop   variable     value
0    ptg000003l        0    10000       skew -0.070141
1    ptg000003l    10000    20000       skew  0.050598
2    ptg000003l    20000    30000       skew -0.042857
3    ptg000003l    30000    40000       skew -0.016842
4    ptg000003l    40000    50000       s

#### Wizualizacja

Dla wielu bakteryjnych genomów bakteryjnych wykres asymetrii GC przyjmuje charakterystyczny kształt, z punktami przełamania odpowiadającym miejscom inicjacji i terminacji replikacji (ori). Jest to bardzo interesujący przykład analizy prostych parametrów sekwencji mającej bezpośrednie przełożenie na biologię. 

Dla zainteresowanych tematem:
- [Analyzing genomes with cumulative skew diagrams](https://academic.oup.com/nar/article/26/10/2286/1030593)
- [Asymmetric substitution patterns: a review of possible underlying mutational or selective mechanisms](https://www.sciencedirect.com/science/article/pii/S0378111999002978?via%3Dihub)
- [SkewIT: The Skew Index Test for large-scale GC Skew analysis of bacterial genomes](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7717575)

In [ ]:
for chromname, skewdf_molten in skew_data_final_molten.items():
    skewplot = (ggplot(skewdf_molten, aes(x="start", y="value", color="variable")) + geom_line() +
           theme_minimal() + ylab("") + xlab("Position [bp]") + scale_colour_discrete(labels=["GC skew", "Cumulative GC skew" ]) +
           labs(colour=""))
    skewplot.save(f"skew_plot_molten_{chromname}.png", width=8, height=6)

/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_molten_ptg000007l.png
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_molten_ptg000003l.png
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_molten_ptg000012l.png
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternoteb

### Sformatowanie danych do narzędzia Circos

Dane dotyczące kumulatywnej zawartości GC zwizualizujemy w postaci wykresu liniowego. Format danych dla tego typu wykresu, tak jak w poprzednim przypadku nie ulega zmianie.

In [ ]:
for chromname, skewdf_molten in skew_data_final_molten.items():
    print(skewdf_molten)

            chr    start     stop   variable     value
0    ptg000007l        0    10000       skew  0.099786
1    ptg000007l    10000    20000       skew  0.046055
2    ptg000007l    20000    30000       skew -0.016490
3    ptg000007l    30000    40000       skew -0.039448
4    ptg000007l    40000    50000       skew -0.033190
..          ...      ...      ...        ...       ...
987  ptg000007l  4910000  4920000  skew_cuml  0.379409
988  ptg000007l  4920000  4930000  skew_cuml  0.266545
989  ptg000007l  4930000  4940000  skew_cuml  0.217113
990  ptg000007l  4940000  4950000  skew_cuml  0.160676
991  ptg000007l  4950000  4960000  skew_cuml  0.160676

[992 rows x 5 columns]
            chr    start     stop   variable     value
0    ptg000003l        0    10000       skew -0.070141
1    ptg000003l    10000    20000       skew  0.050598
2    ptg000003l    20000    30000       skew -0.042857
3    ptg000003l    30000    40000       skew -0.016842
4    ptg000003l    40000    50000       s

In [ ]:
c_skew_data_final_molten = {}

for chromname, skewdf_molten in skew_data_final_molten.items():
    #print(skewdf_molten)
    c_skewdf = skewdf_molten[skewdf_molten["variable"] == "skew_cuml"].copy()

    #c_skewdf = skewdf_molten[["chr","start","stop","skew_cuml"]].copy()
    c_skewdf = c_skewdf[["chr", "start", "stop", "value"]]
    c_skewdf = c_skewdf.rename(columns={"value": "skew_cuml"})
    c_skewdf['col'] = "color=pink"
    c_skew_data_final_molten[chromname] = c_skewdf

    c_skewdf = c_skewdf.reset_index(drop=True)

    print(c_skewdf)

            chr    start     stop  skew_cuml         col
0    ptg000007l        0    10000   0.280468  color=pink
1    ptg000007l    10000    20000   0.327354  color=pink
2    ptg000007l    20000    30000   0.310566  color=pink
3    ptg000007l    30000    40000   0.270406  color=pink
4    ptg000007l    40000    50000   0.236617  color=pink
..          ...      ...      ...        ...         ...
491  ptg000007l  4910000  4920000   0.379409  color=pink
492  ptg000007l  4920000  4930000   0.266545  color=pink
493  ptg000007l  4930000  4940000   0.217113  color=pink
494  ptg000007l  4940000  4950000   0.160676  color=pink
495  ptg000007l  4950000  4960000   0.160676  color=pink

[496 rows x 5 columns]
            chr    start     stop  skew_cuml         col
0    ptg000003l        0    10000   0.463790  color=pink
1    ptg000003l    10000    20000   0.533110  color=pink
2    ptg000003l    20000    30000   0.474395  color=pink
3    ptg000003l    30000    40000   0.451321  color=pink
4    pt

In [ ]:
for chromname, c_skewdf in c_skew_data_final_molten.items():
    print(chromname)
    c_skewdf.to_csv(f"gc_skew_cumul_{chromname}.histo", sep="\t", index=False, header=False)

ptg000007l
ptg000003l
ptg000012l
ptg000008l
ptg000002l
ptg000016l
ptg000018l
ptg000014l
ptg000011l
ptg000015l
ptg000010l
ptg000004l
ptg000001l
ptg000009l
ptg000013l
ptg000017l


## Wizualizacja połączeń oraz kafelków na wykresie typu Circos

Aby zapoznać się z elementami typu "links" (połączenia) oraz "tile" (kafelek) na wykresie Circos wykorzystamy pozycje najliczniej wystepującego 20-meru. W tym celu najpierw zliczymy wystąpienia obecnych w sekwencji 20-merów.

In [15]:
kmers_df = {}

for chromname in chromnames:
    seq = fadict[chromname]

    kmers = kmer_freq(seq)
    kmers = sorted(kmers.items(), key=lambda n: n[1], reverse=True)
    print(chromname, kmers[0])
    kmers_df[chromname] = kmers


ptg000007l ('ggaccctgagaccctgatta', 30)
ptg000003l ('ccttagatcatgacagaaat', 16)
ptg000012l ('tctctctctctctctctctc', 37)
ptg000008l ('actcccgatggtcacccaga', 23)
ptg000002l ('ggaccctgagaccctgatta', 15)
ptg000016l ('TCACCATGCTCCGTCGATAT', 35)
ptg000018l ('ggaccctgagaccctgatta', 18)
ptg000014l ('taatcagggtctcagggtcc', 20)
ptg000011l ('gagagagagagagagagaga', 40)
ptg000015l ('tttttttttttttttttttt', 36)
ptg000010l ('aaaaaaaaaaaaaaaaaaaa', 24)
ptg000004l ('tttttttttttttttttttt', 22)
ptg000001l ('actcgtgcactcgatttgac', 41)
ptg000009l ('ATCCACCCTAAACTGATGGG', 24)
ptg000013l ('tgagcttgcagacctcgaaa', 177)
ptg000017l ('tttttttttttttttttttt', 40)


### Znajdowanie Pozycji 20-mera o Największej Częstości

W tej sekcji zidentyfikujemy 20-mer o największej częstości w sekwencji genomu i znajdziemy jego pozycje na genomie.


In [16]:
kpos_df = {}

for chromname, kmers in kmers_df.items():
    kpos_df[chromname] = {}
    print(chromname)
    for chromname_2 in chromnames:
        print("  ", chromname_2)
        seq = fadict[chromname_2]
        max_kmer = ''
        for i in range(len(kmers)):
            print(kmers[i][0])
            if len(list(set(kmers[i][0]))) > 1:
                print(list(set(kmers[i][0])))
                max_kmer = kmers[i][0]
                break
            else:
                continue

        kpos = kmer_pos(seq, max_kmer)
        print(kpos)
        
        kpos_df[chromname][chromname_2] = (kpos, kmers)


ptg000007l
   ptg000007l
ggaccctgagaccctgatta
['t', 'c', 'a', 'g']
[(113340, 113360), (154163, 154183), (178406, 178426), (186143, 186163), (211649, 211669), (292609, 292629), (302045, 302065), (338788, 338808), (345484, 345504), (432950, 432970), (433654, 433674), (686305, 686325), (809577, 809597), (931083, 931103), (962459, 962479), (1050002, 1050022), (1228044, 1228064), (1228589, 1228609), (1235699, 1235719), (1242179, 1242199), (1300928, 1300948), (1366133, 1366153), (1501159, 1501179), (1573823, 1573843), (3309689, 3309709), (3781514, 3781534), (4317533, 4317553), (4755161, 4755181), (4761782, 4761802), (4764741, 4764761)]
   ptg000003l
ggaccctgagaccctgatta
['t', 'c', 'a', 'g']
[(804687, 804707), (832232, 832252), (848904, 848924), (858460, 858480), (884300, 884320), (1016139, 1016159), (1018664, 1018684), (1067855, 1067875), (1432617, 1432637), (1439212, 1439232), (1450598, 1450618), (1456782, 1456802), (1588375, 1588395), (1609796, 1609816), (1627735, 1627755)]
   ptg000012l
g

#### Połączenie nakładających się zakresów

In [22]:
kpos_merged_df = {}

for chromname in kpos_df.keys():
    print(chromname)

    kpos_merged_df[chromname] = {}

    for chromname_2 in kpos_df[chromname]:
        print("  ", chromname_2)

        kpos = kpos_df[chromname][chromaname_2][0]
        kmers = kpos_df[chromname][chromaname_2][1]

        kpos_merged = merge_overlaps(kpos)
        print(kpos_merged)

        kpos_merged_df[chromname][chromname_2] = kpos_merged


ptg000007l
   ptg000007l
   ptg000003l
   ptg000012l
   ptg000008l
   ptg000002l
   ptg000016l
   ptg000018l
   ptg000014l
   ptg000011l
   ptg000015l
   ptg000010l
   ptg000004l
   ptg000001l
   ptg000009l
   ptg000013l
   ptg000017l
ptg000003l
   ptg000007l
   ptg000003l
   ptg000012l
   ptg000008l
   ptg000002l
   ptg000016l
   ptg000018l
   ptg000014l
   ptg000011l
   ptg000015l
   ptg000010l
   ptg000004l
   ptg000001l
   ptg000009l
   ptg000013l
   ptg000017l
ptg000012l
   ptg000007l
   ptg000003l
   ptg000012l
   ptg000008l
   ptg000002l
   ptg000016l
   ptg000018l
   ptg000014l
   ptg000011l
   ptg000015l
   ptg000010l
   ptg000004l
   ptg000001l
   ptg000009l
   ptg000013l
   ptg000017l
ptg000008l
   ptg000007l
   ptg000003l
   ptg000012l
   ptg000008l
   ptg000002l
   ptg000016l
   ptg000018l
   ptg000014l
   ptg000011l
   ptg000015l
   ptg000010l
   ptg000004l
   ptg000001l
   ptg000009l
   ptg000013l
   ptg000017l
ptg000002l
   ptg000007l
   ptg000003l
   ptg000012l
   ptg0

#### Przygotowanie Danych do Circos

Teraz, gdy mamy pozycje 20-merów, które chcemy wyświetlić w Circos, przygotujemy dane, które będą zgodne z tym narzędziem do wizualizacji.
Pozycje 20-merów, które uwidocznimy w postaci kafelków sformatujemy w sposób tożsamy z poprzednimi danymi.


In [ ]:
for chromname, kpos_merged in kpos_merged_df.items(): 
    with open(f"max_kmer_{chromname}.histo", "w") as out:
        col = "darkblue"
        for line in kpos_merged:
            print(line)
            lineout = [chromname] + list(line) + [f"color={col}"]
            print(lineout)
            out.write("\t".join([str(x) for x in lineout]) + "\n")

(113340, 113360)
['ptg000007l', 113340, 113360, 'color=darkblue']
(154163, 154183)
['ptg000007l', 154163, 154183, 'color=darkblue']
(178406, 178426)
['ptg000007l', 178406, 178426, 'color=darkblue']
(186143, 186163)
['ptg000007l', 186143, 186163, 'color=darkblue']
(211649, 211669)
['ptg000007l', 211649, 211669, 'color=darkblue']
(292609, 292629)
['ptg000007l', 292609, 292629, 'color=darkblue']
(302045, 302065)
['ptg000007l', 302045, 302065, 'color=darkblue']
(338788, 338808)
['ptg000007l', 338788, 338808, 'color=darkblue']
(345484, 345504)
['ptg000007l', 345484, 345504, 'color=darkblue']
(432950, 432970)
['ptg000007l', 432950, 432970, 'color=darkblue']
(433654, 433674)
['ptg000007l', 433654, 433674, 'color=darkblue']
(686305, 686325)
['ptg000007l', 686305, 686325, 'color=darkblue']
(809577, 809597)
['ptg000007l', 809577, 809597, 'color=darkblue']
(931083, 931103)
['ptg000007l', 931083, 931103, 'color=darkblue']
(962459, 962479)
['ptg000007l', 962459, 962479, 'color=darkblue']
(1050002, 

### Wizualizacja połączeń (links)

Połączenia w obrębie wykresów Circos wykorzystywane są m.in. do wizualizacji regionów homologicznych. Tutaj jako przykład wykorzystamy znane nam pozycje najliczniejszego 20-meru.

Połączenia definiujemy określając łączone pary zakresów w pojedynczych liniach:

|chrom1|start1|stop1|chrom2|start2|stop2|parametry
|--|--|--|--|--|--|--|
|	0|	169475|	169522	|	0|	224535	|224555	|color=blue
|	0|	33243	|35664	|	1|	442	|816	|color=blue


In [ ]:
for chromname, kpos_merged in kpos_merged_df.items():
    # Create an empty DataFrame to store the results
    kmer_links_df = pd.DataFrame(columns=["Chromname1", "Start1", "Stop1","Chromname2", "Start2", "Stop2","Color"])
    col = "blue"

    for i in range(len(kpos_merged)):
        for j in range(i,len(kpos_merged)):
            if i != j:
                lineout = [chromname, *kpos_merged[i], chromname, *kpos_merged[j], f"color={col}"]
                kmer_links_df.loc[len(kmer_links_df)] = lineout

    print(kmer_links_df)
    kmer_links_df.to_csv(f"max_kmer_{chromname}.links", sep="\t", index=False, header=False)

     Chromname1   Start1    Stop1  Chromname2   Start2    Stop2       Color
0    ptg000007l   113340   113360  ptg000007l   154163   154183  color=blue
1    ptg000007l   113340   113360  ptg000007l   178406   178426  color=blue
2    ptg000007l   113340   113360  ptg000007l   186143   186163  color=blue
3    ptg000007l   113340   113360  ptg000007l   211649   211669  color=blue
4    ptg000007l   113340   113360  ptg000007l   292609   292629  color=blue
..          ...      ...      ...         ...      ...      ...         ...
430  ptg000007l  4317533  4317553  ptg000007l  4761782  4761802  color=blue
431  ptg000007l  4317533  4317553  ptg000007l  4764741  4764761  color=blue
432  ptg000007l  4755161  4755181  ptg000007l  4761782  4761802  color=blue
433  ptg000007l  4755161  4755181  ptg000007l  4764741  4764761  color=blue
434  ptg000007l  4761782  4761802  ptg000007l  4764741  4764761  color=blue

[435 rows x 7 columns]
     Chromname1   Start1    Stop1  Chromname2   Start2    Stop2 

In [ ]:
# Write the DataFrame to a file
kmer_links_df.to_csv("max_kmer.links", sep="\t", index=False, header=False)

### Pliki konfiguracyjne Circos

Aby stworzyć wykres, oprócz danych, musimy przygotować kilka plików konfiguracyjnych, w których definiujemy sposób w jaki wyświetlone mają być nasze dane.

Są to:

#### Kariotyp

Określamy w nim podstawowe informacje dotyczące segmentów (chromosomów), które chcemy zwizualizować. Format:

|chr|- |ID (w danych)|etykieta|początek|koniec|kolor 
|--|--|--|--|--|--|--|
|chr|	-|	1|	chromosome1|	0|	580076|	purple|

W tym przykładzie kariotyp jest mało skomplikowany, i możemy go w prosty sposób wygenerować korzystając z dostepnych danych.


In [ ]:
karyo_df = {}

for chromname in chromnames:
    seq = fadict[chromname]
    seq_length = len(seq)

    karyo = ["chr","-",chromname,chromname,"0",str(seq_length),"purple"]
    print(karyo)
    karyo_df[chromname] = karyo

['chr', '-', 'ptg000007l', 'ptg000007l', '0', '4962671', 'purple']
['chr', '-', 'ptg000003l', 'ptg000003l', '0', '1805757', 'purple']
['chr', '-', 'ptg000012l', 'ptg000012l', '0', '2044213', 'purple']
['chr', '-', 'ptg000008l', 'ptg000008l', '0', '2324804', 'purple']
['chr', '-', 'ptg000002l', 'ptg000002l', '0', '2259285', 'purple']
['chr', '-', 'ptg000016l', 'ptg000016l', '0', '2349775', 'purple']
['chr', '-', 'ptg000018l', 'ptg000018l', '0', '2524214', 'purple']
['chr', '-', 'ptg000014l', 'ptg000014l', '0', '2808376', 'purple']
['chr', '-', 'ptg000011l', 'ptg000011l', '0', '3109985', 'purple']
['chr', '-', 'ptg000015l', 'ptg000015l', '0', '3330951', 'purple']
['chr', '-', 'ptg000010l', 'ptg000010l', '0', '4100071', 'purple']
['chr', '-', 'ptg000004l', 'ptg000004l', '0', '3736295', 'purple']
['chr', '-', 'ptg000001l', 'ptg000001l', '0', '3711849', 'purple']
['chr', '-', 'ptg000009l', 'ptg000009l', '0', '4221094', 'purple']
['chr', '-', 'ptg000013l', 'ptg000013l', '0', '4262825', 'purp

In [ ]:
with open("karyo.conf", "w") as out:
    for chromname, karyo in karyo_df.items():
        out.write("\t".join(karyo)+"\n")


#### Główny plik konfiguracyjny

Definiujemy w nim elementy grafiki, oraz ich parametry. W głównym pliku konfiguracyjnym określamy lokalizację pliku z kariotypem:

>  karyotype = karyo.conf

Elementami, które możemy uwzględnić na wykresie są m.in.:

Ideogramy, czyli segmenty "chromosomów":

> \<ideogram\>
> 
> \<spacing\> default = 0.005r \</spacing\>
> 
> radius           = 0.90r 
> thickness        = 20p 
> fill             = yes
> 
> #stroke_thickness = 1
> #stroke_color     = black
> 
> 
> \</ideogram\>

Oznaczenia osi (ticks):

> \<tick> spacing        = 10000 u
>  color          = grey 
>  size           = 10p
>   \</tick>

 Wykresy prezentujące dane liczbowe:

> \<plot>
> 
> type = histogram
>  file = gc_content.histo
>   thickness = 0p
> 
> \</plot>

Linie łączące elementy wykresu:

> \<links> 
> \<link> radius = 0.8r 
> bezier_radius = 0r 
> bezier_radius_purity = 0.9
>  color = black 
>  thickness = 2 
>  file = max_kmer.links 
>  \</link> 
>  \</links>

Plik konfiguracyjny do tego ćwiczenia możemy pobrać z chmury:

In [ ]:
!wget "https://drive.google.com/uc?export=download&id=1eWXYScnQnOAyd50cI3ycwpigDJFxlvSv" -O circos_conf.conf
!ls | grep conf

18736.38s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


zsh:1: command not found: wget


18741.76s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


karyo.conf


In [ ]:
!cat circos_conf.conf


########## Kariotyp - tutaj definiujemy chromosomy
karyotype = karyo.conf



<ideogram>

<spacing>
# spacing between ideograms
default = 0.005r
</spacing>

# ideogram position, thickness and fill
radius           = 0.90r
thickness        = 20p
fill             = yes

#stroke_thickness = 1
#stroke_color     = black


</ideogram>


####### Ta część w zasadzie jest skopiowana z pliku przykładowego:

<image>
<<include etc/image.conf>> # included from Circos distribution
radius* = 1500 
</image>


# RGB/HSV color definitions, color lists, location of fonts,
# fill patterns
<<include etc/colors_fonts_patterns.conf>> # included from Circos distribution

# debugging, I/O an dother system parameters
<<include etc/housekeeping.conf>> # included from Circos distribution

#<ticks> blocks to define ticks, tick labels and grids
#
# requires that chromosomes_units be defined
#
 

### Znaczniki na osiach:
<<include ticks.conf>>



<plots>


#### GC-content
<plot>

type = histogram
file = gc_content.h

#### Dodatkowe pliki konfiguracyjne 

Możemy definiować w nich te same typy elementów, co w głównym pliku konfiguracyjnym. Uwzględniamy je w głównym pliku linijką \<\<include name.conf\>\>.



In [ ]:
!wget "https://drive.google.com/uc?export=download&id=11ZgDy-rKfUbjYmsZOSfnFJLAwBcm-C5_" -O ticks.conf
!ls | grep conf

--2025-10-28 13:15:25--  https://drive.google.com/uc?export=download&id=11ZgDy-rKfUbjYmsZOSfnFJLAwBcm-C5_
Resolving drive.google.com (drive.google.com)... 216.58.208.206, 2a00:1450:401b:800::200e
Connecting to drive.google.com (drive.google.com)|216.58.208.206|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=11ZgDy-rKfUbjYmsZOSfnFJLAwBcm-C5_&export=download [following]
--2025-10-28 13:15:26--  https://drive.usercontent.google.com/download?id=11ZgDy-rKfUbjYmsZOSfnFJLAwBcm-C5_&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.38.161, 2a00:1450:401b:802::2001
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.38.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 499 [application/octet-stream]
Saving to: ‘ticks.conf’

ticks.conf          100%[===================>]     499  --.-KB/s    in 0s      

2025-10

In [ ]:
!cat ticks.conf


show_ticks          = yes
show_tick_labels    = yes

<ticks>
skip_first_label = no
skip_last_label = no
radius           = dims(ideogram,radius_outer)
multiplier       = 1 
color            = black
thickness        = 2p
size             = 20p

<tick>
skip_first_label = no
spacing        = 100000u
show_label     = yes
label_size     = 20p
label_offset   = 10p
format         = %d
suffix = " bp"
</tick>

<tick>
spacing        = 10000 u
color          = grey
size           = 10p
</tick>

</ticks>


### Wywołanie Circos


In [ ]:
!circos -conf circos_conf.conf

/bin/bash: line 1: circos: command not found
